# S02: choose a model for a bounded business task

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/01_model_and_serving.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. Question: which model can reliably interpret a spend request, and what evidence would justify the serving choice? The database, rather than model arithmetic, owns the amount.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–10 min | Establish the business result.** Orders store integer paise. Nova has two AsterWorks Q1 orders totalling INR 65,000. The model should extract quarter and supplier only. Changing the tenant in natural language must never change server identity.

In [ ]:
db=Purchases()
args={'quarter':'2026-Q1','supplier':'Nova'}
truth=db.summary(FINANCE,args)
assert truth['total_minor']==6500000 and truth['order_count']==2
print(truth)
print('INR',truth['total_minor']/100)

**10–25 min | Inspect the contract and failures.** These are authored fixtures, not model outputs. Explain why unsupported fields and missing quarter produce a clarification rather than a guessed total. Then run the real experiment if LIVE is enabled.

In [ ]:
from engine.purchasing import QUERY_PROMPT
fixtures=read_data('model_response_fixtures.json')
print(json.dumps(fixtures,indent=2))
for candidate in [args, {'quarter':None,'supplier':'Nova'}, {'quarter':'2026-Q1','supplier':'Nova','tenant':'beacon'}]:
    print(candidate, '=>', query_errors(candidate) or db.summary(FINANCE,candidate))

**25–45 min | Compare models on the same cases.** Enter a second model ID through `CHAT_MODEL_B` if the endpoint supports it. Development cases can guide prompt edits; freeze the prompt before looking at holdout results. Six cases are a classroom demonstration, not production confidence. Repeat stochastic runs only within the approved budget. Observe correctness, latency and token usage separately.

In [ ]:
cases=read_data('query_cases.json')
rows=[]
if LIVE:
    ids=list(dict.fromkeys([client.model,os.environ.get('CHAT_MODEL_B',client.model)]))
    for model_id in ids:
        candidate=ModelClient.environment(model=model_id)
        for case in cases:
            before=len(candidate.calls)
            try:
                actual=candidate.json(QUERY_PROMPT,case['question'])
                passed=actual==case['expected']
                outcome={'parsed':actual,'matches_label':passed}
            except Exception as exc:
                outcome={'error_type':type(exc).__name__,'matches_label':False}
            rows.append({'model':model_id,'case':case['id'],'split':case['split'],**outcome,
                         'telemetry':candidate.calls[-1] if len(candidate.calls)>before else {}})
    print(json.dumps(rows,indent=2))
    Path('outputs').mkdir(exist_ok=True)
    Path('outputs/model-experiment.json').write_text(json.dumps(rows,indent=2))
else:
    print('No model ranking produced. Use this time to agree labels and inspect the SQL boundary.')

**45–60 min | Serving calculation and decision.** Parameter memory alone is not total serving memory. This calculation is illustrative. KV cache, concurrency, activations, runtime overhead and availability replicas can dominate. A cloud GPU benchmark and approved prices are needed for a deployment decision.

In [ ]:
parameters=8_000_000_000
for bytes_per_parameter in [2,1,0.5]:
    print(bytes_per_parameter,'bytes/parameter => weights only GiB:',round(parameters*bytes_per_parameter/2**30,2))
# A hypothetical workload, not a measured throughput claim.
requests_per_second=2; mean_service_seconds=4
print('Mean in-flight requests (stable-system estimate):',requests_per_second*mean_service_seconds)
db.close()

**60–70 min | Deliver an ADR.** State: target task; observed model results or pending evidence; hosted/self-hosted preference; reasons; data location and availability constraints; cost assumptions; test that could reverse the choice. Explain when prompting, RAG and fine-tuning address different problems. A larger model is not automatically the right answer.

**Change exercise:** add a request for an unsupported quarter and a ambiguous supplier. Decide how the product should ask for clarification without inventing a query. Do not change the holdout labels to make a model pass.